# Paper Metadata

One tidy row per publication. Primary key: `paper_id` (Dimensions `pub.…`, shared with every other
output in `Dimensions/output/`). The Dimensions twin of `OpenAlex/notebook/paper_metadata.ipynb`;
column names are kept where the two agree.

## Input — the parts `references_w_year.ipynb` wrote, nothing from the dump
```
Dimensions/cache/pub_scalars/part_*.parquet   # year, type, class, source, ref_count, FoR, citation counts, n_authors, doi
Dimensions/cache/pub_authors/part_*.parquet   # author_list, team_size, first/last/corresponding author, countries
Dimensions/cache/paper_journal.parquet        # built here by dim.build_journal(): source_id, journal, is_journal
Dimensions/cache/paper_fos.parquet            # built here by dim.build_fos(): FoS_rep, FoS_0, FoR codes
```

## Output columns
| column | | OpenAlex twin |
|---|---|---|
| `paper_id`, `year` | | same |
| `doctype` | Dimensions `type`: article, chapter, proceeding, preprint, monograph, book | `type` (a different vocabulary) |
| `doc_class`, `is_citable` | `document_type.classification` / `.is_citable` — RESEARCH_ARTICLE, REVIEW_ARTICLE, CONFERENCE_ABSTRACT, EDITORIAL, … | — |
| `ref_count` | references within the dump (`len(reference_ids)`) | count of `referenced_works` rows |
| `journal`, `is_journal`, `source_id` | `source_titles` title; `is_journal` = `source_titles.type == 'journal'` | `sources.type == 'journal'` |
| `author_list`, `team_size` | resolved researcher ids in author order; all author slots | `author_list` (null there), `paper_author.team_size` |
| `FoS_rep`, `FoS_0` | ANZSRC FoR 2020 **division** (23): first listed; all, `;`-joined | OpenAlex field (26) |
| `for_division_codes`, `for_group_codes` | the 2- and 4-digit codes | — |
| `cited_by_count`, `citations_count` | `metrics.times_cited`, `citations_count` from the dump (Dimensions' own count, not the graph's) | `cited_by_count` |
| `doi` | | — |

`domain` and `is_retracted` have no Dimensions source and are not written.

Output → `Dimensions/output/paper_metadata.parquet`, sorted by `paper_id`.

In [1]:
import os, sys, gc, glob, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Dimensions')
import dim_common as dim
ROOT = dim.BASE; OUT = dim.OUT
print('dump:', dim.ROOT)
import duckdb
OUT_FP = f'{OUT}/paper_metadata.parquet'
# Everything below is fed by notebook/references_w_year.ipynb: it writes the per-publication
# map (year + source), the scalar and author parts, and the edge table with both years and both
# source ids. Build it once before running this notebook.
assert dim.have_consolidated(), (
    'run notebook/references_w_year.ipynb first -- it builds the map, the scalar parts and the edge table')
dim.summary()

dump: /project/jevans/dimensions/dimensions/dimensions_june_2025
dump     : /project/jevans/dimensions/dimensions/dimensions_june_2025
cache    : /project/jevans/Dawoon/Science of Science/Dimensions/cache
output   : /project/jevans/Dawoon/Science of Science/Dimensions/output
  consolidated edge table: present
  map            2.80 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/pub_year_source_map.npz
  graph        not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_graph.npz
  csr          not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_csr.npz
  journal      not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_journal.parquet
  fos          not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_fos.parquet
  pat2pub      not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/patent2pub_edges.parquet
  scalars       4219 parts  /project/jevans/Dawoon/Science o

## 1. Journal and field caches

In [2]:
%%time
# 1. publication -> journal (map x source_titles), publication -> field (scalar parts)
dim.build_journal()
dim.build_fos()
con = duckdb.connect()
con.execute("SET memory_limit='200GB'"); con.execute(f"SET temp_directory='{dim.CACHE}/duckdb_tmp'")
con.execute('SET preserve_insertion_order=false')
print(con.execute(f"""SELECT (SELECT count(*) FROM read_parquet('{dim.JOURNAL_PQ}')) AS journal_rows,
                            (SELECT count(*) FROM read_parquet('{dim.JOURNAL_PQ}') WHERE is_journal) AS in_journal,
                            (SELECT count(*) FROM read_parquet('{dim.FOS_PQ}')) AS fos_rows""").df().to_string(index=False))

journal map from pub_year_source_map.npz: 132,452,366 publications with a source, 119,623,106 in a journal-type source -> /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_journal.parquet
FoS map: 119,058,431 publications with a FoR division -> /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_fos.parquet
 journal_rows  in_journal  fos_rows
    132452366   119623106 119058431


## 2. Assemble + save

In [3]:
%%time
# 2. Assemble and save (DuckDB, out-of-core), sorted by paper_id
t0 = time.time()
con.execute(f"""
COPY (
  SELECT s.pub_id AS paper_id, s.year, s.type AS doctype, s.doc_class, s.is_citable,
         s.ref_count::INTEGER AS ref_count,
         j.journal, coalesce(j.is_journal, false) AS is_journal, s.source_id,
         a.author_list, a.team_size::INTEGER AS team_size,
         f.FoS_0, f.FoS_rep, f.for_division_codes, f.for_group_codes,
         s.times_cited AS cited_by_count, s.citations_count, s.doi
  FROM read_parquet('{dim.SCALARS}/part_*.parquet') s
  LEFT JOIN read_parquet('{dim.JOURNAL_PQ}') j ON j.work_id = s.pub_id
  LEFT JOIN read_parquet('{dim.FOS_PQ}')     f ON f.work_id = s.pub_id
  LEFT JOIN read_parquet('{dim.AUTHORS}/part_*.parquet') a ON a.pub_id = s.pub_id
  WHERE s.pid IS NOT NULL
  ORDER BY paper_id
) TO '{OUT_FP}.tmp' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)""")
os.replace(OUT_FP + '.tmp', OUT_FP)
n = con.execute(f"SELECT count(*), count(DISTINCT paper_id) FROM read_parquet('{OUT_FP}')").fetchone()
print(f'WROTE {OUT_FP}  ({n[0]:,} rows, {n[1]:,} distinct, {os.path.getsize(OUT_FP)/1e9:.2f} GB) in {time.time()-t0:.0f}s')
assert n[0] == n[1], 'paper_id is not unique'
print(con.execute(f"""
SELECT round(100.0*count(journal)/count(*),1) AS pct_journal_title, round(100.0*count(*) FILTER (WHERE is_journal)/count(*),1) AS pct_is_journal,
       round(100.0*count(FoS_rep)/count(*),1) AS pct_fos, round(100.0*count(author_list)/count(*),1) AS pct_author_list,
       round(100.0*count(*) FILTER (WHERE ref_count > 0)/count(*),1) AS pct_with_refs
FROM read_parquet('{OUT_FP}')""").df().to_string(index=False))

WROTE /project/jevans/Dawoon/Science of Science/Dimensions/output/paper_metadata.parquet  (155,507,994 rows, 155,507,994 distinct, 5.22 GB) in 178s
 pct_journal_title  pct_is_journal  pct_fos  pct_author_list  pct_with_refs
              84.9            76.9     76.6             91.6           51.2


## 3. Example

In [4]:
# 3. Shape — rows per type and per decade, and ten example rows (never the whole file)
display(con.execute(f"""SELECT doctype, count(*) AS n, round(100.0*count(*)/sum(count(*)) OVER (),1) AS pct,
                        round(avg(ref_count),1) AS mean_refs, round(100.0*count(FoS_rep)/count(*),1) AS pct_fos
                        FROM read_parquet('{OUT_FP}') GROUP BY 1 ORDER BY n DESC""").df())
display(con.execute(f"""SELECT (year//10)*10 AS decade, count(*) AS n, round(avg(ref_count),1) AS mean_refs,
                        round(100.0*count(*) FILTER (WHERE is_journal)/count(*),1) AS pct_journal
                        FROM read_parquet('{OUT_FP}') WHERE year BETWEEN 1900 AND 2025 GROUP BY 1 ORDER BY 1""").df())
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') WHERE ref_count > 5 AND FoS_rep IS NOT NULL LIMIT 10").df())
con.close()

,doctype,n,pct,mean_refs,pct_fos
0,article,124013142,79.7,14.6,80.1
1,chapter,15865478,10.2,12.1,46.6
2,proceeding,8929126,5.7,6.6,80.0
3,preprint,4673274,3.0,11.3,95.8
4,monograph,1122724,0.7,22.6,32.7
5,book,901630,0.6,5.9,32.9
6,seminar,2620,0.0,1.5,94.7


,decade,n,mean_refs,pct_journal
0,1900,611778,0.2,86.3
1,1910,671694,0.3,85.8
2,1920,924232,0.6,87.8
3,1930,1208077,1.0,91.8
4,1940,1250308,1.1,93.9
5,1950,2709901,1.7,93.2
6,1960,4527782,2.8,90.6
7,1970,6982774,4.4,89.7
8,1980,10079121,6.5,87.4
9,1990,14903372,8.7,83.1


,paper_id,year,doctype,doc_class,is_citable,ref_count,journal,is_journal,source_id,author_list,team_size,FoS_0,FoS_rep,for_division_codes,for_group_codes,cited_by_count,citations_count,doi
0,pub.1003194778,2011,article,RESEARCH_ARTICLE,True,8,Journal of the Korean Society of Food Science ...,True,jour.1117510,ur.014706530671.42;ur.07576420471.12;ur.011407...,8,"Agricultural, Veterinary and Food Sciences","Agricultural, Veterinary and Food Sciences",30,3006,15,15,10.3746/jkfn.2011.40.4.481
1,pub.1003194779,2013,article,RESEARCH_ARTICLE,True,42,PLOS ONE,True,jour.1037553,ur.01044413022.50;ur.01207007063.25;ur.0713203...,7,Biological Sciences;Chemical Sciences,Biological Sciences,31;34,3101,20,20,10.1371/journal.pone.0054104
2,pub.1003194782,2015,article,RESEARCH_ARTICLE,True,40,Research on Aging,True,jour.1090099,ur.01237370603.41;ur.013333642001.17;ur.010550...,7,Biomedical and Clinical Sciences;Psychology,Biomedical and Clinical Sciences,32;52,3202;5202;5204,28,28,10.1177/0164027515574777
3,pub.1003194783,2013,article,RESEARCH_ARTICLE,True,30,International Scholarly Research Notices,True,jour.1429544,ur.013040313062.57;ur.013446640601.66,2,Economics,Economics,38,3801;3803,0,<NA>,10.1155/2013/761482
4,pub.1003194784,2013,article,RESEARCH_ARTICLE,True,19,Modelling and Simulation in Materials Science ...,True,jour.1035090,ur.010065361755.07;ur.010332710054.26;ur.01447...,3,Engineering,Engineering,40,4016;4017,30,30,10.1088/0965-0393/22/1/015012
5,pub.1003194785,2014,article,None,<NA>,10,British Journal of Dermatology,True,jour.1000902,ur.01013143460.87,1,Biomedical and Clinical Sciences,Biomedical and Clinical Sciences,32,3202,1,1,10.1111/bjd.13256
6,pub.1003194787,2008,article,RESEARCH_ARTICLE,True,12,Computers in Entertainment,True,jour.1140851,ur.013634265673.06;ur.015656303235.06;ur.01542...,3,Information and Computing Sciences,Information and Computing Sciences,46,4607,7,7,10.1145/1324198.1324206
7,pub.1003194788,2012,article,RESEARCH_ARTICLE,True,98,Journal of Biotechnology,True,jour.1294801,ur.01073775742.05;ur.01114102276.87;ur.0727626...,3,"Agricultural, Veterinary and Food Sciences;Bio...","Agricultural, Veterinary and Food Sciences",30;32;36,3001;3206;3601,49,49,10.1016/j.jbiotec.2012.03.009
8,pub.1003194791,2010,article,RESEARCH_ARTICLE,True,18,Environmental Chemistry Letters,True,jour.1033923,ur.012130040462.20;ur.01153432205.90;ur.010435...,3,Chemical Sciences,Chemical Sciences,34,3401;3406,72,72,10.1007/s10311-010-0289-8
9,pub.1003194793,2005,article,RESEARCH_ARTICLE,True,11,Journal of Geodesy,True,jour.1052480,ur.015675271723.14;ur.015463731302.77;ur.07775...,3,Earth Sciences,Earth Sciences,37,3706,105,105,10.1007/s00190-005-0003-y
